In [ ]:
import os

# Export these values in the kernel environment before running this notebook.
required_environment = ['SAGEMAKER_BUCKET', 'SAGEMAKER_EFS_ID', 'SAGEMAKER_IMAGE_URI', 'SAGEMAKER_OUTPUT_S3_URI', 'SAGEMAKER_ROLE_ARN', 'SAGEMAKER_SECURITY_GROUP_IDS', 'SAGEMAKER_SUBNET_IDS']
missing = [name for name in required_environment if not os.environ.get(name, "").strip()]
if missing:
    raise ValueError("Configure required environment variables: " + ", ".join(missing))


# SageMaker Rebalance Probe

Lanza un job corto que no entrena el modelo. Solo construye el `TrafficDataset`, calcula scores de severidad por `time_idx`, rebalancea `train_time_indices` y exporta un resumen antes/después.


In [ ]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session, s3_path_join

sagemaker_session = Session(default_bucket=os.environ["SAGEMAKER_BUCKET"])
region = sagemaker_session.boto_region_name
role = os.environ["SAGEMAKER_ROLE_ARN"]
bucket_name = os.environ["SAGEMAKER_BUCKET"]
print(region)
print(role)


In [ ]:
import os
import tarfile
import tempfile
from pathlib import Path
import boto3

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'

def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    source_path = os.path.abspath(source_path)
    with tempfile.TemporaryDirectory() as tmpdir:
        tar_path = os.path.join(tmpdir, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_path, 'w:gz') as tar:
            tar.add(os.path.join(source_path, 'src'), arcname='src')
            tar.add(os.path.join(source_path, 'requirements-sagemaker.txt'), arcname='requirements-sagemaker.txt')
        s3 = boto3.client('s3', region_name=region)
        s3_uri = f's3://{bucket}/{s3_key_prefix}/{TAR_SOURCE_FILENAME}'
        s3.upload_file(tar_path, bucket, f'{s3_key_prefix}/{TAR_SOURCE_FILENAME}')
        return s3_uri

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/rebalance-probe/code')
print(source_code_uri)


In [ ]:
image = os.environ["SAGEMAKER_IMAGE_URI"]
job_name = f'congestion-bogota-rebalance-probe-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}'

hyper_parameters = {
    'sagemaker_program': 'src/training_entrypoint_rebalanced_probe.py',
    'sagemaker_submit_directory': source_code_uri,
    'input-seq-len': '150',
    'output-seq-len': '50',
    'node-batch-size': '500',
    'train-ratio': '0.8',
    'include-weather': 'True',
    'include-velocity-target': 'True',
    'use-minimal-datasets': 'True',
    'rebalance-method': 'future_high_fraction',
    'high-level-threshold': '3.0',
    'bucket-thresholds': '0.005,0.012',
    'topk-fraction': '0.1',
    'rebalance-target-ratios': '0.5,0.25,0.25',
    'rebalance-rng-seed': '42',
}

print(job_name)


In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=region)

NUMERIC_REGEX = r"([-+0-9eE\\.]+)"

metric_definitions_payload = [
    {"Name": "before_low_ratio", "Regex": rf"before_low_ratio={NUMERIC_REGEX}"},
    {"Name": "before_medium_ratio", "Regex": rf"before_medium_ratio={NUMERIC_REGEX}"},
    {"Name": "before_high_ratio", "Regex": rf"before_high_ratio={NUMERIC_REGEX}"},
    {"Name": "after_low_ratio", "Regex": rf"after_low_ratio={NUMERIC_REGEX}"},
    {"Name": "after_medium_ratio", "Regex": rf"after_medium_ratio={NUMERIC_REGEX}"},
    {"Name": "after_high_ratio", "Regex": rf"after_high_ratio={NUMERIC_REGEX}"},
    {"Name": "score_min", "Regex": rf"score_min={NUMERIC_REGEX}"},
    {"Name": "score_max", "Regex": rf"score_max={NUMERIC_REGEX}"},
    {"Name": "score_mean", "Regex": rf"score_mean={NUMERIC_REGEX}"},
    {"Name": "train_windows_before", "Regex": rf"train_windows_before={NUMERIC_REGEX}"},
    {"Name": "train_windows_after", "Regex": rf"train_windows_after={NUMERIC_REGEX}"},
    {"Name": "val_windows", "Regex": rf"val_windows={NUMERIC_REGEX}"},
]

training_job_request = {
    "TrainingJobName": job_name,
    "RoleArn": role,
    "AlgorithmSpecification": {
        "TrainingImage": image,
        "TrainingInputMode": "File",
        "MetricDefinitions": metric_definitions_payload,
        "EnableSageMakerMetricsTimeSeries": True,
    },
    "HyperParameters": hyper_parameters,
    "InputDataConfig": [
        {
            "ChannelName": "train",
            "DataSource": {
                "FileSystemDataSource": {
                    "FileSystemId": os.environ["SAGEMAKER_EFS_ID"],
                    "FileSystemAccessMode": "ro",
                    "FileSystemType": "EFS",
                    "DirectoryPath": "/data/datos_procesados",
                }
            },
        },
    ],
    "OutputDataConfig": {
        "S3OutputPath": os.environ["SAGEMAKER_OUTPUT_S3_URI"],
    },
    "ResourceConfig": {
        "InstanceType": "ml.r7i.8xlarge",
        "InstanceCount": 1,
        "VolumeSizeInGB": 100,
    },
    "StoppingCondition": {
        "MaxRuntimeInSeconds": 7200,
    },
    "VpcConfig": {
        "SecurityGroupIds": [v.strip() for v in os.environ["SAGEMAKER_SECURITY_GROUP_IDS"].split(",") if v.strip()],
        "Subnets": [v.strip() for v in os.environ["SAGEMAKER_SUBNET_IDS"].split(",") if v.strip()],
    },
}

response = sm.create_training_job(**training_job_request)
print(response["TrainingJobArn"])


In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=region)
desc = sm.describe_training_job(TrainingJobName=job_name)
print(desc["TrainingJobStatus"])
print(desc["SecondaryStatus"])
print(desc.get("FailureReason"))


In [ ]:
desc = boto3.client('sagemaker', region_name=region).describe_training_job(TrainingJobName=job_name)
for metric in desc.get('FinalMetricDataList', []):
    print(metric['MetricName'], metric['Value'])
